In [ ]:
import pandas as pd
from Bio import AlignIO

# Read the input TSV files
dr_pairs_df = pd.read_csv('data/IN_filtered_DR_100pairs.tsv', sep='\t')
print (dr_pairs_df.head())

#read unreduced
with open('data/in.fullseq', 'r') as file:
    in_unre_seq = file.read().splitlines()
# Print the aligned sequences
# for sequence in in_unre_seq[:2]:
#     print(sequence)

with open('data/in.reduce4.seq', 'r') as file:
    in_re_seq = file.read().splitlines()
# Print the aligned sequences
# for sequence in in_re_seq[:2]:
#     print(sequence)

    # Read the redux file and create the dictionary
def get_redu_dict(redux_file, offset):
    redux_dict = {}
    with open(redux_file, 'r') as file:
        for line in file:
            parts = line.strip().split()
            position = int(parts[0])+offset
            groups = parts[1:]
            for i, group in enumerate(groups):
                if group == '-':
                    continue
                key = (position, chr(65 + i))  # 'A', 'B', 'C', 'D', etc.
                redux_dict[key] = list(group)
    return redux_dict
#get the reduced pair in format like 'C140D' translate back to unreduced with rules that 1. look up in dictionary ('C', 140)get the last letter in list, and samely, look up in dictionary ('D', 140), get the last letter in list, and combine the new translated unreduced two letters put them back to new pairs, like 'G140S'

def reduced_to_unreduced(redux_dict, reduced_pair):
    # Parse the reduced pair
    wildtype, pos, mutate = reduced_pair[0], int(reduced_pair[1:-1]), reduced_pair[-1]
    # Look up the unreduced wildtype and mutate in the dictionary
    unreduced_wildtype = redux_dict.get((pos, wildtype), ['-'])[-1]
    unreduced_mutate = redux_dict.get((pos, mutate), ['-'])[-1]
    
    # Combine the new translated unreduced letters with the position
    unreduced_pair = f"{unreduced_wildtype}{pos}{unreduced_mutate}"
    return unreduced_pair

def calculate_freq(aa, pos, seq_list):
    count = sum(1 for seq in seq_list if seq[pos - 1] == aa)
    frequency = count / len(seq_list)
    return frequency


  wildtype_aa1  pos1 mutate_aa1 wildtype_aa2  pos2 mutate_aa2      ddE  \
0            C   140          D            D   148          B  8.50879   
1            D   143          A            D   230          C  6.37754   
2            B   157          A            D   160          B  6.08820   
3            C   140          A            D   148          A  5.61299   
4            C   140          D            D   148          C  5.20420   

   Mutant_MSA_frequency1(based_on_unreduced)  \
0                                   0.199180   
1                                   0.023770   
2                                   0.064754   
3                                   0.011475   
4                                   0.199180   

   Mutant_MSA_frequency2(based_on_unreduced) Unreduced_DRM_with_freq_lt_0.01  
0                                   0.186066                     C140D;D148B  
1                                   0.022951                     D143A;D230C  
2                            

In [17]:
in_dict = get_redu_dict('data/in.reduce4.redux', 1)
print(reduced_to_unreduced(in_dict,'D148B'))
#use the ditionry, to ranslate all dr_pairs_df reduce seuqnce back to unreduced

Q148H


In [18]:
# Create a new DataFrame with the desired columns
new_df = dr_pairs_df[['ddE']].copy()

# Add the 'MutPair(reduced)' column by concatenating the relevant columns
new_df['MutPair(reduced)'] = dr_pairs_df.apply(
    lambda row: f"{row['wildtype_aa1']}{row['pos1']}{row['mutate_aa1']}-{row['wildtype_aa2']}{row['pos2']}{row['mutate_aa2']}",
    axis=1
)

new_df['MutPair(unreduced)'] = new_df['MutPair(reduced)'].apply(
    lambda reduced: '-'.join([reduced_to_unreduced(in_dict, pair) for pair in reduced.split('-')])
)

# Display the new DataFrame
print(new_df.head())
print(calculate_freq('H',148,in_unre_seq))

#now add the following columns : FreqMut1(unreduced;reduced), FreqMut2(unreduced;reduced). start from MutPair(reduced) and MutPair(unreduced) column, where get the unreduced Mutant frequency in unreduced seq and reduced Mutant freq in reduced seq, do the same for the second Mut pair 

# Add the 'FreqMut1(unreduced;reduced)' and 'FreqMut2(unreduced;reduced)' columns
new_df['FreqMut1(unreduced;reduced)'] = new_df.apply(
    lambda row: f"{calculate_freq(row['MutPair(unreduced)'].split('-')[0][-1], int(row['MutPair(unreduced)'].split('-')[0][1:-1]), in_unre_seq):.3f};"
                f"{calculate_freq(row['MutPair(reduced)'].split('-')[0][-1], int(row['MutPair(reduced)'].split('-')[0][1:-1]), in_re_seq):.3f}",
    axis=1
)

new_df['FreqMut2(unreduced;reduced)'] = new_df.apply(
    lambda row: f"{calculate_freq(row['MutPair(unreduced)'].split('-')[1][-1], int(row['MutPair(unreduced)'].split('-')[1][1:-1]), in_unre_seq):.3f};"
                f"{calculate_freq(row['MutPair(reduced)'].split('-')[1][-1], int(row['MutPair(reduced)'].split('-')[1][1:-1]), in_re_seq):.3f}",
    axis=1
)
print(new_df.head())

       ddE MutPair(reduced) MutPair(unreduced)
0  8.50879      C140D-D148B        D140Q-Q148H
1  6.37754      D143A-D230C        V143W-I230R
2  6.08820      B157A-D160B        E157F-E160G
3  5.61299      C140A-D148A        D140H-Q148N
4  5.20420      C140D-D148C        D140Q-Q148L
0.1860655737704918


IndexError: string index out of range